# Demo of the Q.ANT's NPU with MNIST digit recognition

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import qant_native_computing_toolkit.ai as q_ai
import qant_native_computing_toolkit.generic as q_generic
import utils

### Prepare input and neural network data


1. Load a random image from the MNIST test data set and weights for a trained two-layer neural network

In [ ]:
img, img_org, label = utils.load_random_image()
n_colors, img_size_x, img_size_y = img.shape
_ = plt.imshow(img_org.squeeze(), cmap="gray_r")

print(img.shape)
weights_layer_1, bias_layer_1, weights_layer_2, bias_layer_2 = utils.load_NN_weights()
features = img.flatten().reshape((img.shape[0], img.shape[1] * img.shape[2]))

2. Upload the model weights

In [ ]:
# upload weights for layer 1
w1_handle = q_generic.upload(
    weights_layer_1,
    data_role=q_generic.DataRole.Weights,
    device_id=0,
)
# allocate result buffer for layer 1 (optional)
r1_handle = q_generic.allocate(
    features.shape[0],
    weights_layer_1.shape[0],
    data_role=q_generic.DataRole.Features,
    device_id=0,
)

# upload weights for layer 2
w2_handle = q_generic.upload(
    weights_layer_2,
    data_role=q_generic.DataRole.Weights,
    device_id=0,
)

3. Perform the inference on the NPU and display the result

In [ ]:
def inference(features, bias_layer_1, bias_layer_2):
    # execute layer 1 (with implicit upload of features & download of result)
    fm = q_ai.linear_fprop_preloaded_weights(features, w1_handle)

    # remainder of layer 1
    fm = q_ai.add_bias_fprop(fm, bias_layer_1)
    fm = q_ai.relu_fprop(fm)

    # upload features explicitly to NPU
    q_generic.upload_into(fm, r1_handle)

    # execute layer 2 (with implicit upload of features & download of result)
    # if you want to pre-uploaded features, you can use q_ai.linear_fprop_preloaded
    fm = q_ai.linear_fprop_preloaded_weights(fm, w2_handle)

    fm = q_ai.add_bias_fprop(fm, bias_layer_2)
    res = q_ai.softmax_fprop(fm).squeeze()

    prediction = np.argmax(res)
    certainty = np.max(res) * 100
    return prediction, certainty


prediction, certainty = inference(features, bias_layer_1, bias_layer_2)
utils.display_result(prediction, certainty)